# 01 Session Explorer

Sync one session locally, explore processed artifacts, and inspect optional raw chunks for debugging.

In [ ]:
from pathlib import Path
import json
import logging
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

NOTEBOOK_ROOT = Path.cwd()
LIB_PATH = NOTEBOOK_ROOT / "lib"
if str(LIB_PATH) not in sys.path:
    sys.path.insert(0, str(LIB_PATH))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")
load_dotenv(NOTEBOOK_ROOT / ".env")
sns.set_theme(style="whitegrid")

from remote_sync import get_latest_session, get_session_by_id
from session_loader import (
    load_clean_streams,
    load_raw_chunks,
    load_session_summary,
    load_window_features,
)

In [ ]:
# Parameters
user_id = 2
session_id = None  # Set explicit session ID or keep None for latest
max_raw_rows_per_stream = 500

In [ ]:
if session_id:
    session_cache_path = get_session_by_id(
        user_id=user_id,
        session_id=session_id,
        local_cache_root=NOTEBOOK_ROOT / "data_cache",
        env_path=NOTEBOOK_ROOT / ".env",
    )
else:
    session_cache_path = get_latest_session(
        user_id=user_id,
        local_cache_root=NOTEBOOK_ROOT / "data_cache",
        env_path=NOTEBOOK_ROOT / ".env",
    )

print(f"Synced session cache: {session_cache_path}")

In [ ]:
clean_streams = load_clean_streams(session_cache_path)
window_features = load_window_features(session_cache_path)
summary = load_session_summary(session_cache_path)
raw_streams = load_raw_chunks(
    session_cache_path,
    max_rows_per_stream=max_raw_rows_per_stream,
)

print(f"Clean streams: {sorted(clean_streams.keys())}")
print(f"Window streams: {sorted(window_features.keys())}")
print(f"Raw streams: {sorted(raw_streams.keys())}")

## Session Summary

In [ ]:
if summary:
    print(json.dumps(summary, indent=2, ensure_ascii=False))
else:
    print("No session summary found.")

## Window Counts Per Stream

In [ ]:
rows = []
for stream_type, df in window_features.items():
    rows.append({
        "stream_type": stream_type,
        "window_count": len(df),
        "columns": ",".join(df.columns.astype(str).tolist()),
    })

pd.DataFrame(rows).sort_values("stream_type") if rows else pd.DataFrame()

## HR Quick Plots (If Available)

In [ ]:
def _pick_column(columns, candidates):
    lookup = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return None

hr_clean = clean_streams.get("hr")
hr_windows = window_features.get("hr")

if hr_clean is not None and not hr_clean.empty:
    ts_col = _pick_column(hr_clean.columns, ["ts_utc", "timestamp", "received_at_collector"])
    hr_col = _pick_column(hr_clean.columns, ["hr", "heart_rate"])
    if ts_col and hr_col:
        plot_df = hr_clean.copy()
        plot_df[ts_col] = pd.to_datetime(plot_df[ts_col], errors="coerce")
        plot_df = plot_df.dropna(subset=[ts_col]).sort_values(ts_col)

        plt.figure(figsize=(12, 4))
        sns.lineplot(data=plot_df, x=ts_col, y=hr_col)
        plt.title("HR Over Time")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()

        plt.figure(figsize=(8, 4))
        sns.histplot(plot_df[hr_col].dropna(), bins=30)
        plt.title("HR Histogram")
        plt.tight_layout()
        plt.show()
    else:
        print("HR clean stream found but expected columns are missing.")
else:
    print("No HR clean stream available.")

if hr_windows is not None and not hr_windows.empty:
    ts_col = _pick_column(hr_windows.columns, ["window_start_ts", "window_start", "ts_utc"])
    mean_col = _pick_column(hr_windows.columns, ["mean_hr", "window_mean_hr", "hr_mean"])
    if ts_col and mean_col:
        trend_df = hr_windows.copy()
        trend_df[ts_col] = pd.to_datetime(trend_df[ts_col], errors="coerce")
        trend_df = trend_df.dropna(subset=[ts_col]).sort_values(ts_col)

        plt.figure(figsize=(12, 4))
        sns.lineplot(data=trend_df, x=ts_col, y=mean_col)
        plt.title("HR Window Mean Trend")
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.show()
    else:
        print("HR window features found but expected trend columns are missing.")

## Multi-Stream Notes

In [ ]:
known_plot_streams = {"hr"}
for stream_type, df in clean_streams.items():
    if stream_type in known_plot_streams:
        continue
    print(f"Stream '{stream_type}': {len(df)} rows, columns={list(df.columns)}")

## Raw Data Debug View

In [ ]:
raw_rows = []
for stream_type, payload in raw_streams.items():
    raw_rows.append({
        "stream_type": stream_type,
        "chunk_count": payload.get("total_lines", 0),
        "loaded_preview_rows": payload.get("loaded_rows", 0),
        "first_ts": payload.get("first_timestamp"),
        "last_ts": payload.get("last_timestamp"),
        "is_truncated": payload.get("is_truncated", False),
    })

raw_df = pd.DataFrame(raw_rows).sort_values("stream_type") if raw_rows else pd.DataFrame()
raw_df

In [ ]:
for stream_type, payload in raw_streams.items():
    records = payload.get("records", [])
    sample = records[0] if records else {}
    print(f"\n[{stream_type}] raw path: {payload.get('path')}")
    print(f"sample payload keys: {list(sample.keys()) if isinstance(sample, dict) else []}")
    print(json.dumps(sample, indent=2, ensure_ascii=False)[:2000])

In [ ]:
raw_set = set(raw_streams.keys())
processed_set = set(clean_streams.keys()) | set(window_features.keys())

missing_raw = sorted(processed_set - raw_set)
missing_processed = sorted(raw_set - processed_set)

if missing_raw:
    print(f"WARNING: processed artifacts exist without matching raw data for streams: {missing_raw}")
if missing_processed:
    print(f"WARNING: raw data exists without matching processed artifacts for streams: {missing_processed}")
if not missing_raw and not missing_processed:
    print("Raw and processed stream sets are aligned for discovered streams.")

In [ ]:
print("Raw artifact paths:")
for stream_type, payload in sorted(raw_streams.items()):
    print(f"  {stream_type}: {payload.get('path')}")

print("\nProcessed clean artifact paths:")
for stream_type, df in sorted(clean_streams.items()):
    print(f"  {stream_type}: rows={len(df)}")

print("\nProcessed window_features artifact paths:")
for stream_type, df in sorted(window_features.items()):
    print(f"  {stream_type}: rows={len(df)}")

if summary:
    print(f"\nSession summary path: {summary.get('_source_path')}")